In [0]:
# The purpose of this notebook is to download the silver data that has been exported to S3 weekly by the
# main databricks workspace. This will enable training of models. 

# Station status is stored in parquet format on S3, organized into daily folders. This folder only
# downloads days of station status that are not yet in our silver table.
# Station information is one small file, replaced on every export, so it is downloaded every time.

from pathlib import Path

import boto3 # This is the Amazon S3 python library
from botocore import UNSIGNED
from botocore.config import Config

BUCKET_NAME = "cuny-sps-citibike-archive"
VOLUME = "/Volumes/workspace/default/citibike" # where the downloaded files are kept

# A client without credentials: the exports folder of the bucket is public
s3 = boto3.client("s3", region_name="us-west-2", config=Config(signature_version=UNSIGNED))

STATUS_TABLE = "workspace.default.silver_station_status" 
INFO_TABLE = "workspace.default.silver_station_info"

# production and dev flags. For dev we won't download anything, just
# check the files and list them

dbutils.widgets.dropdown("run_mode", "dev", ["dev", "production"])
RUN_MODE = dbutils.widgets.get("run_mode")
IS_PRODUCTION = RUN_MODE == "production"
print("Run mode:", RUN_MODE)

In [0]:
# Station status: download the day files that are new, then rebuild the table from all of them.

# List every file in the station_status folder of the bucket.
# The paginator asks again if there are more than 1000 files.

pages = s3.get_paginator("list_objects_v2").paginate(Bucket=BUCKET_NAME, Prefix="exports/station_status/")
s3_file_names = [file["Key"] for page in pages for file in page.get("Contents", [])]

# Keep only the data files. Spark's bookkeeping files start with "_".

data_file_names = [name for name in s3_file_names
                   if name.endswith(".parquet") and not Path(name).name.startswith("_")]

# Download each file that the volume does not have yet. A day file never changes after its upload.
downloaded_count = 0
for s3_file_name in data_file_names:
    local_file = Path(VOLUME) / s3_file_name.removeprefix("exports/")
    if local_file.exists():
        continue
    # Don't download if not in production
    if not IS_PRODUCTION:
        print("dev, not downloaded:", s3_file_name)
        continue
    local_file.parent.mkdir(parents=True, exist_ok=True)
    s3.download_file(BUCKET_NAME, s3_file_name, str(local_file))
    downloaded_count += 1
print(f"{len(data_file_names)} status files in S3; {downloaded_count} new files downloaded")

# Spark reads all day folders as one table, and takes nyc_date from the folder names
if IS_PRODUCTION:
    spark.sql(f"""
        CREATE OR REPLACE TABLE {STATUS_TABLE} AS
        SELECT * FROM parquet.`{VOLUME}/station_status`
    """)

In [0]:
# Station information: download the file every time, because each export replaces it.

if IS_PRODUCTION:
    s3.download_file(BUCKET_NAME, "exports/station_info.parquet", f"{VOLUME}/station_info.parquet")

    spark.sql(f"""
        CREATE OR REPLACE TABLE {INFO_TABLE} AS
        SELECT * FROM parquet.`{VOLUME}/station_info.parquet`
    """)
else:
    print("dev, not downloaded: exports/station_info.parquet")

In [0]:
# Check: the newest day in each table, and the number of rows.

display(spark.sql(f"""
    SELECT 'station status' AS table_name, COUNT(*) AS rows, MAX(nyc_date) AS newest_day
    FROM {STATUS_TABLE}
    UNION ALL
    SELECT 'station information', COUNT(*), to_date(MAX(fetched_at))
    FROM {INFO_TABLE}
"""))